In [ ]:
from langgraph_sdk import get_client
from dotenv import load_dotenv
import os

load_dotenv()

In [ ]:
# Set LANGSMITH_API_KEY in your environment
client = get_client(
    # url="https://cc-lsd-deep-agent-assistant-15de37b3b13051ee9f72140dbca7989a.us.langgraph.app",
    url="http://127.0.0.1:2024",

    api_key=os.environ["LANGSMITH_API_KEY"],
)
print("🔗 Connected to LangGraph server")

In [ ]:
async def stream_response(thread_id: str, assistant_id: str, message: str) -> None:
    """Stream an assistant's response to the console."""
    last_seen = {}
    async for chunk in client.runs.stream(
        thread_id,
        assistant_id,
        input={"messages": [{"role": "human", "content": message}]},
        stream_mode="messages",
    ):
        if chunk.event == "messages/partial":
            for msg in chunk.data:
                if msg.get("type") == "ai":
                    mid = msg.get("id", "")
                    text = next((b["text"] for b in msg.get("content", []) if b.get("type") == "text"), "")
                    print(text[len(last_seen.get(mid, "")):], end="", flush=True)
                    last_seen[mid] = text
    print()

In [ ]:
# 2. Create a new assistant configuration
GRAPH_ID = "deep_agent"
print("🤖 Creating a new assistant configuration...")

assistant = await client.assistants.create(
    graph_id = GRAPH_ID,
    config={
        "configurable": {
            "system_prompt": "You are a helpful AI assistant. Your memories are loaded from middleware and are stored in /memories/AGENTS.md. If no /memories/AGENTS.md exists, you can create one to store memories. Always reply in the voice of a pirate from the 1600s.",
            "model": "anthropic:claude-haiku-4-5",
            "selected_tools": ["get_todays_date", "advanced_research"],
            "name": "Pirate"
        }
    },
    name="Pirate",
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")
print(f"   🧠 Model: {assistant['config']['configurable']['model']}")
print(f"   🛠️  Tools: {', '.join(assistant['config']['configurable']['selected_tools'])}")

In [ ]:
# 2. Create a new assistant configuration
print("🤖 Creating a new assistant configuration...")

assistant = await client.assistants.create(
    graph_id = GRAPH_ID,
    config={
        "configurable": {
            "system_prompt": "You are a helpful AI assistant. Your memories are loaded from middleware and are stored in /memories/AGENTS.md. If no /memories/AGENTS.md exists, you can create one to store memories. Always reply in the voice of a cowboy from the Old West.",
            "model": "anthropic:claude-haiku-4-5",
            "selected_tools": ["get_todays_date", "advanced_research", "finance_research"],
            "name": "Cowboy"
        }
    },
    name="Cowboy"
)

print("✅ Assistant created successfully!")
print(f"   📍 Assistant ID: {assistant['assistant_id']}")
print(f"   📝 Name: {assistant['name']}")
print(f"   🔢 Version: {assistant['version']}")
print(f"   🧠 Model: {assistant['config']['configurable']['model']}")
print(f"   🛠️  Tools: {', '.join(assistant['config']['configurable']['selected_tools'])}")

In [ ]:
# Store AGENTS.md in the LangGraph persistent store for each assistant
# pirate/AGENTS.md → Pirate Assistant
# cowboy/AGENTS.md → Cowboy Assistant
#
# StoreBackend stores /memories/AGENTS.md as:
#   namespace = [assistant_id, "filesystem"]
#   key       = "AGENTS.md"
# This appears in LangGraph Studio as: assistant_id/filesystem/AGENTS.md

from datetime import datetime, timezone

now = datetime.now(timezone.utc).isoformat()

# Read AGENTS.md files from disk
with open("../src/assistants/pirate/AGENTS.md") as f:
    pirate_memory = f.read()

with open("../src/assistants/cowboy/AGENTS.md") as f:
    cowboy_memory = f.read()

# Map assistant name → AGENTS.md content
agents_md_by_name = {
    "Pirate": pirate_memory,
    "Cowboy": cowboy_memory,
}

all_assistants = await client.assistants.search(graph_id=GRAPH_ID)
print(f"📋 Found {len(all_assistants)} assistant(s) for graph '{GRAPH_ID}'\n")

for asst in all_assistants:
    name = asst["name"]
    if name not in agents_md_by_name:
        continue

    content = agents_md_by_name[name]
    store_value = {
        "content": content.split("\n"),
        "created_at": now,
        "modified_at": now,
    }

    namespace = [asst["assistant_id"], "filesystem"]
    await client.store.put_item(namespace, key="/AGENTS.md", value=store_value)

    item = await client.store.get_item(namespace, key="AGENTS.md")
    print(f"✅ Stored AGENTS.md for '{name}'")
    print(f"   └─ Namespace : {namespace}")
    # print(f"   └─ Key       : {item['key']}")
    # print(f"   └─ Lines     : {len(item['value']['content'])}")
    # print(f"   └─ Updated   : {item['updated_at']}\n")


In [ ]:
thread = await client.threads.create()
await stream_response(thread["thread_id"], assistant["assistant_id"], "Remember that the name of your horse is moonshine")

In [ ]:
await stream_response(thread["thread_id"], assistant["assistant_id"], "list any files and folders that you have access to")

In [ ]:
await stream_response(thread["thread_id"], assistant["assistant_id"], "Create a file named hello.txt and write 'Hello, world!' in it")

In [ ]:
await stream_response(thread["thread_id"], assistant["assistant_id"], "list any files and folders that you have access to")

In [ ]:
# New thread does not see files created in old thread, but still has access to AGENTS.md memory
thread = await client.threads.create()
await stream_response(thread["thread_id"], assistant["assistant_id"], "list any files and folders that you have access to")